In [ ]:
import os
import json
from glob import glob
from collections import defaultdict

import numpy as np
from PIL import Image, UnidentifiedImageError

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

import matplotlib.pyplot as plt

IMG_SIZE      = (300, 300)
BATCH_SIZE    = 32
EPOCHS_PHASE1 = 30
EPOCHS_PHASE2 = 60
LEARNING_RATE = 1e-4

DINO_FOLDER    = "dataset/dinosaur"
MODEL_PATH     = "models/stage2_dino_species.keras"
CLASS_MAP_PATH = "models/stage2_dino_classes.json"

In [ ]:
%pip install tensorflow numpy pillow scikit-learn matplotlib

In [ ]:
def get_base_class(folder_name):
    return folder_name.split("_")[0]

grouped_folders = defaultdict(list)

for sub in os.listdir(DINO_FOLDER):
    full_path = os.path.join(DINO_FOLDER, sub)
    if os.path.isdir(full_path):
        base_class = get_base_class(sub)
        grouped_folders[base_class].append(full_path)

class_names  = sorted(grouped_folders.keys())
class_to_idx = {cls: i for i, cls in enumerate(class_names)}
idx_to_class = {i: cls for cls, i in class_to_idx.items()}

In [ ]:
print(f"Унікальних класів: {len(class_names)}")
print()
for cls, paths in sorted(grouped_folders.items()):
    print(f"  {cls}: {len(paths)} папки")

In [ ]:
os.makedirs("models", exist_ok=True)
with open(CLASS_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump(idx_to_class, f, indent=2, ensure_ascii=False)

In [ ]:
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def get_images_from_folder(folder):
    paths = glob(f"{folder}/**/*.*", recursive=True)
    return [p for p in paths if os.path.splitext(p)[1].lower() in VALID_EXT]

train_paths = []  # list of (image_path, label)
val_paths   = []
test_paths  = []

for cls, folders in grouped_folders.items():
    label = class_to_idx[cls]

    if len(folders) == 1:
        # image-level split — щоб клас потрапив і в val
        images = get_images_from_folder(folders[0])
        if len(images) < 2:
            train_paths += [(p, label) for p in images]
        else:
            tr_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)
            train_paths += [(p, label) for p in tr_imgs]
            val_paths   += [(p, label) for p in val_imgs]
        continue

    train_f, temp_f = train_test_split(folders, test_size=0.3, random_state=42)

    if len(temp_f) > 1:
        val_f, test_f = train_test_split(temp_f, test_size=0.5, random_state=42)
    else:
        val_f  = temp_f
        test_f = []

    for f in train_f:
        train_paths += [(p, label) for p in get_images_from_folder(f)]
    for f in val_f:
        val_paths   += [(p, label) for p in get_images_from_folder(f)]
    for f in test_f:
        test_paths  += [(p, label) for p in get_images_from_folder(f)]

print(f"Train: {len(train_paths)} зображень")
print(f"Val:   {len(val_paths)} зображень")
print(f"Test:  {len(test_paths)} зображень")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def prefilter_paths(paths_labels):
    """Пробує реально декодувати кожне зображення через TF і відкидає ті, що падають."""
    valid = []
    skipped = 0
    for path, label in paths_labels:
        try:
            raw = tf.io.read_file(path)
            tf.image.decode_image(raw, channels=3, expand_animations=False)
            valid.append((path, label))
        except Exception:
            skipped += 1
    if skipped:
        print(f"  Пропущено {skipped} несумісних зображень")
    return valid

def parse_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img.set_shape([*IMG_SIZE, 3])
    return img, label

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(40 / 360, fill_mode="nearest"),
    layers.RandomTranslation(0.2, 0.2, fill_mode="nearest"),
    layers.RandomZoom(0.3, fill_mode="nearest"),
    layers.RandomBrightness(0.3),
], name="augmentation")

In [ ]:
print("Pre-filtering зображень...")
train_paths = prefilter_paths(train_paths)
val_paths   = prefilter_paths(val_paths)
test_paths  = prefilter_paths(test_paths)

train_img_paths = [p for p, _ in train_paths]
train_labels    = [l for _, l in train_paths]
val_img_paths   = [p for p, _ in val_paths]
val_labels      = [l for _, l in val_paths]
test_img_paths  = [p for p, _ in test_paths]
test_labels     = [l for _, l in test_paths]

y_train = np.array(train_labels, dtype=np.int32)

print(f"\nГотово (шляхи, без RAM):")
print(f"  Train: {len(train_img_paths)}")
print(f"  Val:   {len(val_img_paths)}")
print(f"  Test:  {len(test_img_paths)}")

In [ ]:
class_counts = {cls: 0 for cls in class_to_idx.keys()}
for label in y_train:
    class_counts[idx_to_class[label]] += 1

sorted_items = sorted(class_counts.items(), key=lambda x: -x[1])
bar_labels = [item[0] for item in sorted_items]
bar_counts = [item[1] for item in sorted_items]

fig, ax = plt.subplots(figsize=(max(20, len(bar_labels) * 0.2), 6))
ax.bar(range(len(bar_labels)), bar_counts, color="#4C72B0")
ax.set_xticks(range(len(bar_labels)))
ax.set_xticklabels(bar_labels, rotation=60, ha="right", fontsize=7)
ax.set_ylabel("Кількість зображень")
ax.set_title(f"Баланс класів (train) — {len(bar_labels)} класів")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def augment_and_preprocess(img, label):
    img = augmentation(img, training=True)
    img = preprocess_input(img)
    return img, label

def preprocess_only(img, label):
    img = preprocess_input(img)
    return img, label

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_img_paths, train_labels))
    .shuffle(len(train_img_paths), reshuffle_each_iteration=True)
    .map(parse_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .map(augment_and_preprocess, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_img_paths, val_labels))
    .map(parse_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .map(preprocess_only, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_img_paths, test_labels))
    .map(parse_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .map(preprocess_only, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
) if test_img_paths else None

print("tf.data pipeline готовий")

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

In [ ]:
base_model = EfficientNetV2S(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(1024, activation="relu")(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

# Feature extractor: base + GAP (без head) — для кешування Phase 1
gap_idx = next(i for i, l in enumerate(model.layers)
               if isinstance(l, layers.GlobalAveragePooling2D))
feat_extractor = tf.keras.Model(
    inputs=model.input,
    outputs=model.layers[gap_idx].output,
    name="feat_extractor"
)
feat_dim = feat_extractor.output_shape[-1]

# Head model — ДІЛИТЬ ВАГИ з повною моделлю, тренується на кешованих фічах
feat_inp = layers.Input(shape=(feat_dim,), name="feat_input")
hx = feat_inp
for layer in model.layers[gap_idx + 1:]:
    hx = layer(hx)
head_model = models.Model(inputs=feat_inp, outputs=hx, name="head_model")

print(f"Класів: {len(class_names)}")
print(f"Параметрів базової моделі: {base_model.count_params():,}")
print(f"Параметрів всього:         {model.count_params():,}")
print(f"Розмір фіч-вектора:        {feat_dim}")

In [ ]:
head_model.compile(
    optimizer=optimizers.Adam(LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

head_model.summary()

In [ ]:
AUG_ROUNDS = 3  # кількість аугментованих проходів базою (різні аугментації)

print(f"Кешування фіч ({AUG_ROUNDS} проходи через базову модель)...")
train_feats, train_lbls = [], []
for r in range(AUG_ROUNDS):
    for imgs, lbls in train_ds:
        train_feats.append(feat_extractor(imgs, training=False).numpy())
        train_lbls.append(lbls.numpy())
    print(f"  Раунд {r + 1}/{AUG_ROUNDS} готово")

val_feats, val_lbls = [], []
for imgs, lbls in val_ds:
    val_feats.append(feat_extractor(imgs, training=False).numpy())
    val_lbls.append(lbls.numpy())

X_feat     = np.concatenate(train_feats)
y_feat     = np.concatenate(train_lbls)
X_feat_val = np.concatenate(val_feats)
y_feat_val = np.concatenate(val_lbls)

feat_train_ds = (
    tf.data.Dataset.from_tensor_slices((X_feat, y_feat))
    .shuffle(len(X_feat))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
feat_val_ds = (
    tf.data.Dataset.from_tensor_slices((X_feat_val, y_feat_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f"Train фічі: {X_feat.shape} (~{X_feat.nbytes / 1024**2:.0f} MB)")
print(f"Val   фічі: {X_feat_val.shape}")

In [ ]:
callbacks_phase1 = [
    EarlyStopping(monitor="val_loss", patience=10, min_delta=0.001, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
]

callbacks_phase2 = [
    EarlyStopping(monitor="val_loss", patience=12, min_delta=0.001, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-7)
]

In [ ]:
print("=== Phase 1: тренуємо head на кешованих фічах ===")
history1 = head_model.fit(
    feat_train_ds,
    validation_data=feat_val_ds,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weights,
    callbacks=callbacks_phase1,
    verbose=2
)

In [ ]:
print("=== Phase 2: fine-tuning (останні 150 шарів розморожені) ===")

for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[-150:]:
    layer.trainable = True

model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    class_weight=class_weights,
    callbacks=callbacks_phase2,
    verbose=2
)

In [ ]:
print("\nVal оцінка:")
model.evaluate(val_ds, verbose=2)

if test_ds is not None:
    print("\nTest оцінка:")
    model.evaluate(test_ds, verbose=2)

In [ ]:
model.save(MODEL_PATH)
print(f"Модель збережена: {MODEL_PATH}")
print(f"Class map збережений: {CLASS_MAP_PATH}")